# Laboratorio 6 — Clasificación de Spam y Ham

**Universidad del Valle de Guatemala**  
Inteligencia Artificial

---

## 1. Introducción teórica

### ¿Qué es spam y qué es ham?

En el contexto del procesamiento de lenguaje natural (NLP) y la clasificación de mensajes, se utilizan los términos **spam** y **ham** para distinguir dos categorías de texto:

| Categoría | Descripción |
|-----------|-------------|
| **Spam**  | Mensajes no deseados, generalmente de naturaleza comercial, fraudulenta o maliciosa. Suelen contener ofertas engañosas, premios falsos, enlaces sospechosos o lenguaje agresivo orientado a la acción inmediata del usuario (e.g., *"¡Llame YA!", "Ganó un premio"*). |
| **Ham**   | Mensajes legítimos, escritos por personas reales con intenciones genuinas. Son conversaciones cotidianas, notificaciones esperadas o comunicaciones normales. |

### ¿Por qué es importante este problema?

El filtrado de spam es uno de los problemas más clásicos y prácticos del aprendizaje automático. Se estima que más del **45 % del tráfico de correo electrónico mundial** corresponde a spam. Los costos asociados incluyen pérdida de productividad, riesgos de seguridad (phishing, malware) y consumo de ancho de banda. Un clasificador robusto requiere:

1. **Representación del texto** — convertir texto crudo en características numéricas.
2. **Preprocesamiento lingüístico** — reducir ruido y normalizar vocabulario.
3. **Modelo de clasificación** — aprender patrones que separen las clases.

En este laboratorio nos enfocaremos en los pasos 1 y 2 mediante un **análisis exploratorio de datos (EDA)** exhaustivo, antes y después del preprocesamiento con NLTK.

### Dataset

Utilizamos el **SMS Spam Collection Dataset**, una colección de 5 572 mensajes SMS etiquetados en inglés, ampliamente utilizada como benchmark en la comunidad de NLP.

---
## 2. Instalación de dependencias

Todas las dependencias necesarias están declaradas en el archivo `requirements.txt` incluido en este repositorio.
Antes de ejecutar el notebook, instálelas desde la terminal con:

```bash
pip install -r requirements.txt
```

| Paquete | Uso en este laboratorio |
|---------|-------------------------|
| `pandas` | Carga y manipulación del dataset |
| `matplotlib` | Visualizaciones base (barras, KDE, torta) |
| `seaborn` | Estilos y paletas de color |
| `nltk` | Tokenización, stopwords y lematización |
| `wordcloud` | Generación de nubes de palabras |
| `jupyter` | Entorno de ejecución del notebook |

---
## 3. Carga del dataset y EDA inicial

### 3.1 Importación de librerías

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

# Configuración visual global
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


### 3.2 Carga del CSV

El archivo usa `;` como separador. Cargamos y verificamos las primeras filas.

In [6]:
df = pd.read_csv('spam_ham.csv', sep=';', names=['label', 'text'], header=0, encoding='latin-1')

print(f"Dimensiones del dataset: {df.shape}")
print(f"\nPrimeras 5 filas:")
df.head()

Dimensiones del dataset: (5565, 2)

Primeras 5 filas:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
print("Tipos de datos:")
print(df.dtypes)
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nDistribución de clases:")
print(df['label'].value_counts())
print(f"\nPorcentaje de cada clase:")
print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

Tipos de datos:
label    object
text     object
dtype: object

Valores nulos:
label    0
text     3
dtype: int64

Distribución de clases:
label
ham       4817
spam       746
ham"""       2
Name: count, dtype: int64

Porcentaje de cada clase:
label
ham       86.56 %
spam      13.41 %
ham"""     0.04 %
Name: proportion, dtype: object


### 3.3 Distribución spam vs ham

Visualizamos el balance de clases mediante un gráfico de barras y un gráfico de torta.

In [ ]:
counts = df['label'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # verde=ham, rojo=spam

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title('Conteo de mensajes por clase', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Cantidad de mensajes')
axes[0].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Torta
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=colors, startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('Proporción spam vs ham', fontsize=14, fontweight='bold')

plt.suptitle('Distribución del dataset', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Observación:** El dataset está **desbalanceado**. Aproximadamente el 87 % de los mensajes son ham y solo el 13 % son spam. Esto es relevante al momento de entrenar un clasificador, ya que un modelo naive que siempre prediga *ham* alcanzaría ~87 % de accuracy sin aprender nada útil. Métricas como **precisión, recall y F1** serán más informativas que la accuracy.

### 3.4 Longitud de los mensajes

Calculamos la longitud (en caracteres y palabras) de cada mensaje y analizamos si difiere entre clases.

In [9]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Estadísticas de longitud por clase (caracteres):")
display(df.groupby('label')['char_count'].describe().round(1))
print("\nEstadísticas de longitud por clase (palabras):")
display(df.groupby('label')['word_count'].describe().round(1))

Estadísticas de longitud por clase (caracteres):


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4815.0,66.9,52.6,2.0,32.0,50.0,87.0,910.0
"ham""""""",1.0,57.0,NaN,57.0,57.0,57.0,57.0,57.0
spam,746.0,138.7,29.5,13.0,132.2,149.0,157.0,224.0



Estadísticas de longitud por clase (palabras):


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4815.0,13.5,10.6,1.0,6.5,10.0,18.0,171.0
"ham""""""",1.0,12.0,NaN,12.0,12.0,12.0,12.0,12.0
spam,746.0,23.8,5.8,2.0,22.0,25.0,28.0,35.0


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes,
                           ['char_count', 'word_count'],
                           ['Número de caracteres', 'Número de palabras']):
    for cls, color in palette.items():
        subset = df[df['label'] == cls][col]
        subset.plot(kind='kde', ax=ax, color=color, linewidth=2.5, label=cls)
        ax.axvline(subset.median(), color=color, linestyle='--', linewidth=1.5,
                   label=f'Mediana {cls} = {subset.median():.0f}')
    ax.set_title(f'Densidad — {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de longitud de mensajes por clase (texto original)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Observación:** Los mensajes de spam tienden a ser **significativamente más largos** que los ham. Esto tiene sentido: los spammers incluyen más información (precios, condiciones, números de teléfono, URLs) para persuadir al receptor. Esta diferencia de longitud puede ser una característica muy útil para un clasificador.

### 3.5 Top 20 palabras más frecuentes (texto original)

Separamos los mensajes por clase y contamos la frecuencia de cada token (sin preprocesar aún).

In [ ]:
def get_top_words(series, n=20):
    """Devuelve los n tokens más frecuentes de una Serie de textos."""
    all_words = ' '.join(series.str.lower()).split()
    return Counter(all_words).most_common(n)

top_ham  = get_top_words(df[df['label'] == 'ham']['text'].dropna())
top_spam = get_top_words(df[df['label'] == 'spam']['text'])

def plot_top_words(top_words, title, color, ax):
    words, counts = zip(*top_words)
    bars = ax.barh(list(reversed(words)), list(reversed(counts)),
                   color=color, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Frecuencia')
    for bar, val in zip(bars, list(reversed(counts))):
        ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
plot_top_words(top_ham,  'Top 20 palabras — HAM  (original)', '#2ecc71', axes[0])
plot_top_words(top_spam, 'Top 20 palabras — SPAM (original)', '#e74c3c', axes[1])

plt.suptitle('Palabras más frecuentes por clase — Texto sin preprocesar', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Observación:** Tanto en spam como en ham dominan palabras funcionales (artículos, preposiciones, pronombres) como *"the", "to", "a", "I"*. Estas palabras aportan poca información discriminativa. El preprocesamiento (eliminación de stopwords) será clave para revelar el vocabulario verdaderamente característico de cada clase.

### 3.6 WordCloud por clase (texto original)

In [ ]:
def make_wordcloud(text_series, title, colormap, ax):
    corpus = ' '.join(text_series.str.lower())
    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap=colormap,
        max_words=150,
        collocations=False
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
make_wordcloud(df[df['label'] == 'ham']['text'].dropna(),  'WordCloud — HAM (original)',  'Greens', axes[0])
make_wordcloud(df[df['label'] == 'spam']['text'], 'WordCloud — SPAM (original)', 'Reds',    axes[1])

plt.suptitle('Nube de palabras por clase — Texto sin preprocesar', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Preprocesamiento con NLTK

### 4.1 Justificación del pipeline

Aplicamos las siguientes transformaciones en orden:

| Paso | Técnica | Justificación |
|------|---------|---------------|
| 1 | **Minúsculas** | Unifica tokens como *"FREE"* y *"free"* en un solo tipo. |
| 2 | **Eliminación de puntuación y números** | Signos como `!`, `$`, `?` y dígitos son ruido para el análisis de vocabulario (aunque podrían ser características en sí mismos para clasificación). |
| 3 | **Tokenización** | Separa el texto en unidades mínimas (palabras). |
| 4 | **Eliminación de stopwords** | Palabras funcionales de alta frecuencia y baja información semántica (*the, is, at, which*…). |
| 5 | **Lematización (Lemmatization)** | Reduce cada palabra a su forma canónica (lema). Se prefiere sobre *stemming* porque produce palabras reales y semánticamente interpretables (e.g., *"running" → "run"*, *"better" → "good"*), lo que facilita la interpretación del análisis. |

> **¿Por qué lematización y no stemming?**  
> El **stemming** es más rápido pero trunca las palabras mecánicamente, a veces produciendo raíces sin significado (e.g., *"flies" → "fli"*). La **lematización** usa un diccionario morfológico y siempre devuelve una palabra válida del idioma, lo que es preferible para análisis exploratorio y para construir representaciones de texto interpretables.

### 4.2 Descarga de recursos NLTK

In [18]:
import nltk

for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print(f"Stopwords cargadas: {len(STOP_WORDS)} palabras")
print(f"Ejemplo de stopwords: {sorted(list(STOP_WORDS))[:15]}")

Stopwords cargadas: 198 palabras
Ejemplo de stopwords: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]


### 4.3 Función de preprocesamiento

In [19]:
def preprocess(text):
    """Pipeline completo: minúsculas → limpieza → tokenización → stopwords → lematización."""
    # Paso 1: minúsculas
    text = text.lower()
    # Paso 2: eliminar puntuación y números
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Paso 3: tokenización
    tokens = word_tokenize(text)
    # Paso 4: eliminar stopwords y tokens de un solo carácter
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    # Paso 5: lematización
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Demostración con un ejemplo de cada clase
ejemplo_ham  = df[df['label'] == 'ham']['text'].iloc[0]
ejemplo_spam = df[df['label'] == 'spam']['text'].iloc[0]

print("=== Ejemplo HAM ===")
print(f"Original  : {ejemplo_ham}")
print(f"Procesado : {preprocess(ejemplo_ham)}")
print()
print("=== Ejemplo SPAM ===")
print(f"Original  : {ejemplo_spam}")
print(f"Procesado : {preprocess(ejemplo_spam)}")

=== Ejemplo HAM ===
Original  : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Procesado : go jurong point crazy available bugis great world la buffet cine got amore wat

=== Ejemplo SPAM ===
Original  : Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Procesado : free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate apply


### 4.4 Aplicar preprocesamiento al dataset completo

In [21]:
print("Procesando mensajes...")
df['text_clean'] = df['text'].fillna('').apply(preprocess)

df['char_count_clean'] = df['text_clean'].str.len()
df['word_count_clean'] = df['text_clean'].str.split().str.len()

print(f"Preprocesamiento completado. {len(df)} mensajes procesados.")
df[['label', 'text', 'text_clean']].head(5)

Procesando mensajes...
Preprocesamiento completado. 5565 mensajes procesados.


,label,text,text_clean
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,dun say early hor already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though


---
## 5. EDA con texto preprocesado

Repetimos los mismos análisis del EDA inicial para observar el impacto del preprocesamiento.

### 5.1 Densidad de longitud de mensajes (texto limpio)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes,
                           ['char_count_clean', 'word_count_clean'],
                           ['Número de caracteres', 'Número de palabras']):
    for cls, color in palette.items():
        subset = df[df['label'] == cls][col]
        subset.plot(kind='kde', ax=ax, color=color, linewidth=2.5, label=cls)
        ax.axvline(subset.median(), color=color, linestyle='--', linewidth=1.5,
                   label=f'Mediana {cls} = {subset.median():.0f}')
    ax.set_title(f'Densidad — {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de longitud de mensajes por clase (texto preprocesado)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Top 20 palabras más frecuentes (texto preprocesado)

In [ ]:
top_ham_clean  = get_top_words(df[df['label'] == 'ham']['text_clean'])
top_spam_clean = get_top_words(df[df['label'] == 'spam']['text_clean'])

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
plot_top_words(top_ham_clean,  'Top 20 palabras — HAM  (preprocesado)', '#2ecc71', axes[0])
plot_top_words(top_spam_clean, 'Top 20 palabras — SPAM (preprocesado)', '#e74c3c', axes[1])

plt.suptitle('Palabras más frecuentes por clase — Texto preprocesado', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.3 WordCloud por clase (texto preprocesado)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
make_wordcloud(df[df['label'] == 'ham']['text_clean'],  'WordCloud — HAM  (preprocesado)', 'Greens', axes[0])
make_wordcloud(df[df['label'] == 'spam']['text_clean'], 'WordCloud — SPAM (preprocesado)', 'Reds',   axes[1])

plt.suptitle('Nube de palabras por clase — Texto preprocesado', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Análisis comparativo: antes vs después del preprocesamiento

Visualizamos lado a lado las nubes de palabras para evidenciar el impacto del preprocesamiento.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

make_wordcloud(df[df['label'] == 'ham']['text'].dropna(),        'HAM  — Original',       'Greens', axes[0][0])
make_wordcloud(df[df['label'] == 'spam']['text'].dropna(),       'SPAM — Original',        'Reds',   axes[0][1])
make_wordcloud(df[df['label'] == 'ham']['text_clean'],  'HAM  — Preprocesado',   'Greens', axes[1][0])
make_wordcloud(df[df['label'] == 'spam']['text_clean'], 'SPAM — Preprocesado',   'Reds',   axes[1][1])

plt.suptitle('Comparación: WordCloud antes y después del preprocesamiento', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Análisis de vocabulario exclusivo y compartido

In [27]:
def vocab(series):
    return set(' '.join(series).split())

vocab_ham  = vocab(df[df['label'] == 'ham']['text_clean'])
vocab_spam = vocab(df[df['label'] == 'spam']['text_clean'])

solo_ham  = vocab_ham  - vocab_spam
solo_spam = vocab_spam - vocab_ham
comun     = vocab_ham  & vocab_spam

print(f"Vocabulario HAM  total : {len(vocab_ham):>6} palabras únicas")
print(f"Vocabulario SPAM total : {len(vocab_spam):>6} palabras únicas")
print(f"Palabras compartidas   : {len(comun):>6}")
print(f"Exclusivas de HAM      : {len(solo_ham):>6}")
print(f"Exclusivas de SPAM     : {len(solo_spam):>6}")

Vocabulario HAM  total :   5837 palabras únicas
Vocabulario SPAM total :   1855 palabras únicas
Palabras compartidas   :    896
Exclusivas de HAM      :   4941
Exclusivas de SPAM     :    959


In [ ]:
# Palabras exclusivas de SPAM con mayor frecuencia
spam_texts  = df[df['label'] == 'spam']['text_clean']
spam_counts = Counter(' '.join(spam_texts).split())

top_excl_spam = [(w, c) for w, c in spam_counts.most_common() if w in solo_spam][:20]

ham_texts  = df[df['label'] == 'ham']['text_clean']
ham_counts = Counter(' '.join(ham_texts).split())

top_excl_ham = [(w, c) for w, c in ham_counts.most_common() if w in solo_ham][:20]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, top, title, color in [
    (axes[0], top_excl_ham,  'Top 20 exclusivas de HAM',  '#2ecc71'),
    (axes[1], top_excl_spam, 'Top 20 exclusivas de SPAM', '#e74c3c')
]:
    if top:
        words, counts = zip(*top)
        ax.barh(list(reversed(words)), list(reversed(counts)), color=color, edgecolor='white')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Frecuencia')

plt.suptitle('Palabras exclusivas de cada clase (preprocesado)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 palabras en común
comun_counts = {w: spam_counts[w] + ham_counts[w] for w in comun}
top_comun = Counter(comun_counts).most_common(15)

words_c, counts_c = zip(*top_comun)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(words_c, counts_c, color='#3498db', edgecolor='white', linewidth=0.8)
ax.set_title('Top 15 palabras compartidas entre HAM y SPAM (preprocesado)', fontsize=13, fontweight='bold')
ax.set_xlabel('Palabra')
ax.set_ylabel('Frecuencia total (HAM + SPAM)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Sección de reflexión

### 8.1 ¿Qué cambios son notorios al comparar el texto original y el preprocesado?

Al comparar las visualizaciones antes y después del preprocesamiento, se observan los siguientes cambios:

1. **Desaparición de stopwords dominantes.** En el texto original, palabras como *"to", "the", "a", "I", "you"* ocupaban los primeros lugares en ambas clases por su altísima frecuencia, enmascarando las diferencias entre spam y ham. Tras el preprocesamiento, estas palabras desaparecen y emergen vocabularios distintivos.

2. **Reducción del tamaño de los mensajes.** La longitud promedio (en palabras) disminuye considerablemente, pero la diferencia relativa entre spam y ham se mantiene, lo que confirma que la longitud es una característica robusta.

3. **Normalización morfológica.** La lematización unifica formas verbales y nominales (*"calling", "called", "calls"* → *"call"*), lo que aumenta la frecuencia de los lemas clave y hace los patrones más evidentes.

4. **Vocabulario más semánticamente rico.** Las nubes de palabras preprocesadas revelan términos con carga semántica real: en spam aparecen *"free", "call", "win", "prize", "claim"*; en ham aparecen *"got", "go", "know", "come", "time"*.

---

### 8.2 ¿Qué palabras tienen en común spam y ham?

Las palabras compartidas corresponden al vocabulario general del inglés cotidiano: verbos comunes (*"call", "go", "get", "know", "come"*), sustantivos neutros (*"day", "time", "home"*) y algunos adverbios. Estas palabras aparecen en ambas clases simplemente porque son parte del lenguaje ordinario y no tienen carga discriminativa por sí solas.

---

### 8.3 ¿Qué palabras son exclusivas de cada clase?

**Exclusivas de SPAM:**  
Palabras como *"free", "win", "prize", "claim", "urgent", "guaranteed", "tone", "txt", "mobile", "cash", "award"* son altamente frecuentes y exclusivas del spam. Reflejan el lenguaje de marketing agresivo y la promesa de beneficios inmediatos.

**Exclusivas de HAM:**  
Palabras como *"home", "morning", "night", "miss", "tomorrow", "week", "feel", "dear"* son típicas de conversaciones personales. Reflejan una comunicación cotidiana, afectiva y temporal.

---

### 8.4 ¿Qué características podrían ser útiles para la clasificación?

A partir del EDA, se identifican las siguientes características potencialmente discriminativas:

| Característica | Razón |
|----------------|-------|
| **Longitud del mensaje (chars/words)** | El spam es notoriamente más largo que el ham. |
| **Presencia de palabras clave de spam** | *"free", "win", "call", "prize", "claim"* aparecen casi exclusivamente en spam. |
| **Frecuencia de signos de exclamación y mayúsculas** | El spam usa más énfasis gráfico para llamar la atención. |
| **Presencia de números de teléfono o URLs** | Muy frecuentes en spam (instrucciones de acción). |
| **TF-IDF sobre vocabulario preprocesado** | Captura la importancia relativa de cada término en cada clase. |
| **Proporción de tokens de stopwords eliminadas** | Alta presencia de stopwords puede indicar ham (conversación más natural). |
| **Vocabulario exclusivo de spam** | Construir una lista negra léxica a partir de los tokens exclusivos identificados. |

> **Conclusión:** El preprocesamiento no solo mejora la calidad de las representaciones de texto, sino que también revela patrones lingüísticos que un clasificador bayesiano, de árboles de decisión o basado en transformers puede aprender eficientemente. El siguiente paso natural en este laboratorio sería vectorizar el texto preprocesado con **Bag of Words** o **TF-IDF** y entrenar un clasificador como **Naive Bayes multinomial** o **Regresión Logística**.

---
## 9. Resumen del dataset preprocesado

In [30]:
print("=== Resumen final del dataset ===")
print(f"Total de mensajes       : {len(df)}")
print(f"  - HAM                 : {(df['label']=='ham').sum()} ({(df['label']=='ham').mean()*100:.1f}%)")
print(f"  - SPAM                : {(df['label']=='spam').sum()} ({(df['label']=='spam').mean()*100:.1f}%)")
print(f"\nLongitud promedio (palabras):")
print(f"  - Original  HAM       : {df[df['label']=='ham']['word_count'].mean():.1f}")
print(f"  - Original  SPAM      : {df[df['label']=='spam']['word_count'].mean():.1f}")
print(f"  - Procesado HAM       : {df[df['label']=='ham']['word_count_clean'].mean():.1f}")
print(f"  - Procesado SPAM      : {df[df['label']=='spam']['word_count_clean'].mean():.1f}")
print(f"\nVocabulario (texto preprocesado):")
print(f"  - Tokens únicos HAM   : {len(vocab_ham)}")
print(f"  - Tokens únicos SPAM  : {len(vocab_spam)}")
print(f"  - Compartidos         : {len(comun)}")
print(f"  - Exclusivos HAM      : {len(solo_ham)}")
print(f"  - Exclusivos SPAM     : {len(solo_spam)}")

# Exportar dataset procesado
df.to_csv('spam_ham_procesado.csv', index=False)
print("\nDataset procesado guardado en: spam_ham_procesado.csv")

=== Resumen final del dataset ===
Total de mensajes       : 5565
  - HAM                 : 4817 (86.6%)
  - SPAM                : 746 (13.4%)

Longitud promedio (palabras):
  - Original  HAM       : 13.5
  - Original  SPAM      : 23.8
  - Procesado HAM       : 7.1
  - Procesado SPAM      : 14.7

Vocabulario (texto preprocesado):
  - Tokens únicos HAM   : 5837
  - Tokens únicos SPAM  : 1855
  - Compartidos         : 896
  - Exclusivos HAM      : 4941
  - Exclusivos SPAM     : 959

Dataset procesado guardado en: spam_ham_procesado.csv
